# Exploratory Data Analysis (EDA)
## Occupancy Prediction for Antalya/Alanya Resort Hotels Using Google Trends

**Prepared by:** Bedirhan Sar  
**Project:** Hotel Occupancy Analysis and Prediction

---

## Project Overview

The goal of this project is to investigate whether digital demand signals can help explain and later support the prediction of **daily hotel occupancy rates** for resort hotels in the Antalya/Alanya region of Turkey.

The business context is important. These hotels do not operate only through direct B2C bookings. A substantial part of demand is shaped by **B2B channels**, such as tour operators and travel agencies. Because of this, a search-interest variable like Google Trends should not be treated as a perfect direct demand measure. Instead, it should be evaluated as a possible **early signal** that may reflect travel intent before actual stays happen.

This project therefore asks the following central question:

**Can Google Trends data provide useful information about future occupancy patterns for Antalya/Alanya resort hotels?**

To answer this question, two main data sources were combined:

1. **Hotel occupancy data**
   - Side Mare Hotel
   - Azura Deluxe

2. **Google Trends data**
   - Germany
   - Netherlands
   - United Kingdom
   - Türkiye

This notebook documents the exploratory phase of the project:
- data preparation,
- data merging,
- exploratory data analysis,
- and initial hypothesis-driven statistical examination.


## Problem Definition and Motivation

Tourism demand is highly seasonal, especially for resort hotels. Occupancy in Antalya/Alanya can be influenced by many factors, including:

- seasonal travel cycles,
- school holiday timing,
- exchange-rate conditions,
- destination interest,
- and the actions of tour operators.

This project focuses on one specific external signal: **Google search interest**.

The intuition is simple:

> If people in relevant source markets begin searching more frequently for Antalya-, Alanya-, or hotel-related vacation terms, this search activity may appear before the actual hotel stay and therefore may contain useful predictive information.

At the same time, this project does not assume that Google Trends fully explains occupancy. Because these hotels rely heavily on agency and operator channels, search behavior may only capture part of the story. That makes the analysis more interesting: even a partial signal could still be useful.


## Data Sources

### 1. Hotel Data

Daily occupancy data was prepared for two hotels:
- **Side Mare Hotel**
- **Azura Deluxe**

The main target variable used in this analysis is:
- `occupancy_rate`

Each row in the hotel data represents one hotel on one date.

### 2. Google Trends Data

Google Trends data was collected for tourism-related keywords across four countries:
- Germany
- Netherlands
- United Kingdom
- Türkiye

The raw Trends structure was initially organized around:
- date,
- country,
- keyword,
- trend score

These values were later transformed into an analysis-ready structure.

---

## Why Data Enrichment Was Necessary

Hotel occupancy data alone tells us the **outcome**, but not necessarily the broader context behind demand fluctuations.

Google Trends was used here as an external enrichment source representing possible signals of:
- destination interest,
- travel planning behavior,
- and vacation intent.

This makes the project stronger analytically, because it moves beyond a single internal hotel dataset and tests whether external behavioral data can improve understanding of occupancy patterns.


In [ ]:
import pandas as pd

side = pd.read_excel("side_mare_daily_occupancy_combined.xlsx")
azura = pd.read_excel("azura_deluxe_daily_occupancy_combined.xlsx")

tr_de = pd.read_excel("combined_trends_germany.xlsx")
tr_nl = pd.read_excel("combined_trends_netherlands.xlsx")
tr_uk = pd.read_excel("combined_trends_united_kingdom.xlsx")
tr_tr = pd.read_excel("combined_trends_turkiye.xlsx")


## Data Preparation Process

The first major technical step in the project was converting several separate files into one consistent analytical structure.

### Main steps performed

#### A. Standardizing the hotel files
The two hotel files were aligned to a shared schema:
- `date`
- `hotel_name`
- `occupancy_rate`

This allowed both hotels to be stacked into one common table.

#### B. Standardizing the Google Trends files
The Google Trends files were cleaned so that they followed a consistent format:
- `date`
- `country`
- `keyword`
- `google_trends_score`

#### C. Converting Google Trends from long format to wide format
Originally, the Trends data contained multiple rows for the same date because each date had:
- multiple countries,
- multiple keywords,
- and multiple trend scores.

For analysis, this structure was pivoted into columns such as:
- `trends_germany_alanya_hotel`
- `trends_turkiye_side_otel`
- `trends_netherlands_hotel_alanya`
- and similar features

This made it possible to place all trend features side by side for a given date.

#### D. Building the master table
Finally, the hotel data and the Trends data were merged on `date`.

The resulting combined table follows this logic:

**one row = one hotel on one date**


## What the Master Table Represents

The master table is the central dataset used throughout the project.

It contains:
- the target variable `occupancy_rate`,
- the explanatory Google Trends features,
- and date-level records for both hotels.

This step is essential because it turns several disconnected files into one clean table that can support:
- descriptive analysis,
- correlation analysis,
- lag analysis,
- and later machine learning models.

Without the master table, the analysis would remain fragmented and difficult to reproduce.


In [ ]:
master = pd.read_excel("hotel_master_table.xlsx", sheet_name="master_table")
master["date"] = pd.to_datetime(master["date"])

master.head()


## Data Quality Checks

Before starting EDA, the reliability of the merged dataset must be checked.

The following quality checks were performed:

### 1. Missing-value inspection
Special attention was given to missing values in the Trends columns.

### 2. Duplicate-row inspection
The dataset was checked for repeated `(date, hotel_name)` combinations.

### 3. Date-range inspection
The overlap between the hotel data period and the Trends data period was reviewed.

### 4. Data-type inspection
The following questions were verified:
- Is `date` stored as a true datetime field?
- Is `occupancy_rate` numeric?
- Are the Trends variables numeric?

These checks are important because later statistical analysis and modeling depend on a consistent structure.


In [ ]:
master.info()
master.isna().sum().sort_values(ascending=False).head(20)


## Initial Observations

The first quality checks led to several useful observations:

- The dataset was overall clean enough to proceed with analysis.
- No duplicate `(date, hotel_name)` rows were found.
- Some Trends columns contained missing values.
- These missing values were not random in a problematic sense; they were mainly caused by partial mismatch between the hotel date range and the Trends collection windows.

This is important, but it is not a critical flaw. The reason is that:
- the source of missingness is understandable,
- it can be documented clearly,
- and it can be handled properly in later stages.


## Research Hypotheses

This EDA phase was guided by several simple but important hypotheses.

### H1
Google Trends features have a measurable relationship with hotel occupancy.

### H2
Lagged Google Trends values are more informative than same-day Google Trends values.

### H3
Not all countries and keywords are equally useful; some country-keyword combinations carry stronger signal than others.

### H4
Occupancy contains strong seasonal structure, so any external variable must be interpreted alongside seasonality.


## Univariate EDA: Understanding Occupancy on Its Own

The first variable examined on its own was the target variable, `occupancy_rate`.

The main questions at this stage were:
- In what range does occupancy usually fall?
- Do the two hotels have similar average behavior?
- Are there very low or very high values that may need interpretation?
- Is seasonality clearly visible?

This step matters because it is difficult to interpret any relationship with Google Trends before first understanding the behavior of the target itself.


In [ ]:
master.groupby("hotel_name")["occupancy_rate"].agg(["count", "mean", "median", "std", "min", "max"])


## First Interpretation of Occupancy Behavior

The initial summary statistics suggested that the two hotels are similar in broad structure but not identical in behavior.

Key observations included:
- Side Mare Hotel had a higher average occupancy level.
- Azura Deluxe showed somewhat larger variation.
- Strong seasonal structure was visible in both hotels.
- There were periods with very low or near-zero occupancy, which may reflect off-season closures, very low demand periods, or operational factors.

Even at this early point, one conclusion became clear:

**Seasonality is a major force in this dataset.**

That means any later interpretation of Google Trends must be made carefully, because both Trends and occupancy can rise in similar periods simply due to seasonal tourism patterns.


## Occupancy Behavior Over Time

Summary statistics are helpful, but time-series data needs to be viewed across time to reveal its real structure.

For that reason, occupancy was also examined through:
- daily time plots,
- hotel-level time comparisons,
- and monthly averaged views for a smoother seasonal picture.

These visuals help answer questions such as:
- When does each hotel enter high season?
- Do the hotels move in similar patterns?
- Are there sudden structural breaks or sharp drops?
- Is the yearly rhythm stable across years?


## Examining Google Trends Features

After examining occupancy, the next step was to inspect the Google Trends variables themselves.

The main purpose here was to understand:
- which keywords show meaningful movement,
- which ones remain near zero too often,
- which features are noisy,
- and which ones visually resemble the seasonal shape of occupancy.

This is important because not every theoretically plausible keyword becomes a useful variable in practice. Some search terms may be too weak, too unstable, or too broad to help the analysis.


## Bivariate EDA: Examining Occupancy Together with Google Trends

To move closer to the research question, the next step was to study occupancy together with each Trends feature.

The first comparison focused on **same-day relationships**.

In simple terms, the question was:

> Is the Google Trends value observed on a given day related to the hotel occupancy observed on that same day?

Two correlation measures were used:

### Pearson correlation
Measures linear association.

### Spearman correlation
Measures rank-based association and can still detect monotonic relationships even when the pattern is not strictly linear.


In [ ]:
from scipy.stats import pearsonr, spearmanr

trend_cols = [c for c in master.columns if c.startswith("trends_")]

rows = []
for c in trend_cols:
    sub = master[["occupancy_rate", c]].dropna()
    if len(sub) > 10 and sub[c].nunique() > 1:
        p = pearsonr(sub["occupancy_rate"], sub[c])
        s = spearmanr(sub["occupancy_rate"], sub[c], nan_policy="omit")
        rows.append({
            "feature": c,
            "pearson_r": p.statistic,
            "pearson_p": p.pvalue,
            "spearman_rho": s.statistic,
            "spearman_p": s.pvalue,
        })

same_day_corr = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
same_day_corr.head(10)


## Interpretation of Same-Day Correlations

The first correlation results suggested the following:

- Same-day relationships were generally **weak to moderate**.
- Google Trends did not behave like a strong direct same-day explanation of occupancy.
- This result was consistent with the business logic of the project.

This is not a disappointing result. In fact, it is realistic.

These are not purely direct-booking hotels. Occupancy may be influenced by:
- pre-arranged agency demand,
- operator allocations,
- package-tour flows,
- and decisions made before the stay date.

So a weak same-day relationship does not mean Google Trends is useless. It means that the timing of the signal matters.


## Why Lag Analysis Matters

In tourism, search behavior and actual hotel stays do not happen at the same moment.

A traveler may:
1. search for Antalya today,
2. make a decision later,
3. book later,
4. and stay at the hotel days or weeks afterward.

Because of that, only looking at same-day relationships would be incomplete.

To account for this timing difference, several lags were tested:
- 7 days,
- 14 days,
- 21 days,
- 28 days.

The idea is simple:
for example, in a 28-day lag test, today’s occupancy is compared with the Trends value from **28 days earlier**.


In [ ]:
trend_df = master[["date"] + trend_cols].drop_duplicates("date").sort_values("date").reset_index(drop=True)

lag_rows = []
for lag in [7, 14, 21, 28]:
    lagged = trend_df.copy()
    lagged[trend_cols] = lagged[trend_cols].shift(lag)
    merged = master[["date", "hotel_name", "occupancy_rate"]].merge(lagged, on="date", how="left")
    for c in trend_cols:
        sub = merged[["occupancy_rate", c]].dropna()
        if len(sub) > 10 and sub[c].nunique() > 1:
            p = pearsonr(sub["occupancy_rate"], sub[c])
            lag_rows.append({
                "feature": c,
                "lag_days": lag,
                "pearson_r": p.statistic,
                "pearson_p": p.pvalue,
            })

lag_corr = pd.DataFrame(lag_rows)
best_by_feature = lag_corr.loc[lag_corr.groupby("feature")["pearson_r"].idxmax()].sort_values("pearson_r", ascending=False)
best_by_feature.head(10)


## Interpretation of the Lag Results

This stage produced one of the most meaningful early findings of the project.

The results showed that:
- several Trends features became more useful when delayed,
- and some lagged versions were clearly stronger than their same-day versions.

One notable example was:
- `trends_turkiye_side_otel` showing a stronger relationship at a **28-day lag**.

This matters for two reasons.

### 1. Theoretical meaning
It supports the idea of a realistic travel timeline:
search interest may appear first, while the hotel stay appears later.

### 2. Practical meaning
It suggests that later models should use Google Trends not only in raw same-day form, but also as **lagged features**.


## Evaluation of the Main Hypotheses

### H1: Google Trends has a relationship with occupancy
**Partially supported.**  
Not all variables showed strong signal, but some country-keyword combinations displayed usable relationships.

### H2: Lagged Google Trends is more useful than same-day Google Trends
**Supported.**  
Several lagged features produced stronger correlations than their same-day versions.

### H3: The strength of the relationship depends on country and keyword
**Supported.**  
Some Türkiye- and Germany-related search terms appeared stronger than many UK-based terms.

### H4: Seasonality is a major force in the data
**Supported.**  
Occupancy clearly follows a seasonal structure, which must be considered in all later interpretation.


## What This Notebook Does Not Try to Do

This notebook is focused on the exploratory and analytical stage of the project.

Its primary role is to document:
- data collection,
- cleaning,
- merging,
- EDA,
- and initial hypothesis-based testing.

It is **not** intended to be the final modeling document.

More advanced tasks such as:
- final model selection,
- full forecasting pipeline construction,
- final feature-importance interpretation,
- and fully optimized predictive evaluation

belong to the later machine learning stage of the project.


## Limitations

Several important limitations should be acknowledged at this stage.

### 1. Google Trends is not direct booking data
It captures search interest, not confirmed reservations.

### 2. The B2B structure may weaken the search signal
Because agency and operator channels are strong, search behavior may explain only part of occupancy.

### 3. Not every country and keyword is equally informative
Some variables may be useful, while others may mostly add noise.

### 4. Correlation is not causation
A variable moving with occupancy does not automatically mean it causes occupancy changes.

### 5. Seasonality can act as a confounder
If both Trends and occupancy rise in the same months, part of the relationship may simply reflect shared seasonal timing.


## Main Takeaway from This EDA Phase

The clearest message from this analysis is the following:

> Google Trends is not a strong same-day explanation of resort hotel occupancy, but some lagged Google Trends features appear to carry useful early signal.

This is valuable because it leads to a balanced conclusion:
- the initial business idea is not rejected,
- but Google Trends is also not overstated,
- and the next modeling phase can now focus on the most promising lagged features rather than using all search variables blindly.


## Next Step

The natural next step after this EDA phase is:

1. select the strongest Google Trends features,
2. create their lagged versions,
3. combine them with calendar variables and past occupancy values,
4. and evaluate predictive performance using time-aware train/test splits.

In that sense, this notebook directly prepares the foundation for the machine learning stage.


## Final Summary

In this project phase:

- daily occupancy data was prepared for two hotels,
- Google Trends data was collected for four countries,
- all datasets were merged into a single master table,
- data quality checks were performed,
- occupancy and Trends variables were explored,
- same-day and lagged relationships were tested,
- and several lagged Google Trends signals were found to be more promising than same-day versions.

Overall, the exploratory results suggest that Google Trends alone is not enough to explain occupancy, but selected lagged search features may still be useful as part of a broader predictive framework.
